In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import numpy as np
import os
import re
import warnings
warnings.filterwarnings("ignore")
from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/Bot/Parsing')
from tqdm import tqdm
from urllib.parse import urljoin, urlparse
import time
!pip install fake_useragent
from fake_useragent import UserAgent
UserAgent().chrome
from multiprocessing import Pool
!pip install google-colab-selenium
import google_colab_selenium as gs
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.7 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


#titles

In [ ]:
def write_title_data(data):
    with open('imdb_movies.csv','a') as f:
        writer = csv.writer(f)
        writer.writerow((
            data['url'],
            data['title'],
            data['orig_title'],
            data['movie_type'],
            data['year'],
            data['age_limit'],
            data['m_time'],
            data['genres'],
            data['episodes'],
            data['imdb'],
            data['metascore'],
            data['epi_refs'],
            data['cast_refs'],
            data['genre_refs'],
            data['award_refs'],
            data['plot']
            ))

In [ ]:
def write_error_log(i, url, error_log):
    with open('error_log.txt', 'a') as f:
        f.write(f'{i} | {url} | {error_log} \n')
    with open('error_urls.txt', 'a') as f:
        f.write(url + '\n')

In [ ]:
def get_movie_data(html, url, ind):
    urls = []
    imdb_url = 'https://www.imdb.com'
    mlist = ['TV Mini Series', 'TV Short', 'TV Movie', 'TV Special', 'TV Series']
    soup = BeautifulSoup(html, 'html.parser', from_encoding='utf-8')
    try:
        elem = soup.find('section', class_='ipc-page-background ipc-page-background--base sc-afa4bed1-0 iMxoKo')
        try:
            title_elem = elem.find('div', class_='sc-70a366cc-0 bxYZmb')
            title = title_elem.find('h1').text.strip()
            try:
                orig_title = title_elem.find('div', class_='sc-ec65ba05-1 fUCCIx').text.split('Original title:')[-1].strip()
            except Exception as e:
                orig_title = ''
            try:
                title_info = title_elem.find('ul').findChildren('li')
                movie_type, year, age_limit, m_time = '', '', '', ''
                if title_info:
                    for info in title_info:
                        info = info.text.strip()
                        if info in mlist:
                            movie_type = info
                        if re.match(r'\(*\d{4}–*\d* *\)*', info):
                            year = info
                        if re.match(r'(\d{1,3}h *\d* *m*i*n*)|(\d{1,4} *mi*n*)', info):
                            m_time = info
                        if info in ['Passed', 'Approved','Unrated'] or re.match('\d{1,2}\+', info)\
                         or (re.match(r'[A-Z]{1,2}(\-|\/)*[A-Z0-9]*\-*[A-Z0-9]*', info) and info not in mlist):
                           age_limit = info
                    if movie_type == '':
                        movie_type = 'Movie'
                else:
                    movie_type, year, age_limit, m_time = ['']*4
            except Exception as e:
                error_log = f'Title info exception: {e}, Title: {title}'
                print(error_log)
                write_error_log(ind, url, error_log)
                movie_type, year, age_limit, m_time = ['']*4
        except Exception as e:
            error_log = f'Title element exception: {e}'
            print(error_log)
            write_error_log(ind, url, error_log)
            title, orig_title, movie_type, year, age_limit, m_time = ['']*6
        try:
            plot_elems = soup.find('div', class_='sc-9a2a0028-6 zHrZh')
            elem_genres = plot_elems.find('div', class_='ipc-chip-list__scroller').find_all('a')
            genres = ', '.join([a.text.strip() for a in elem_genres])
            genre_refs = ', '.join([imdb_url + a['href'] for a in elem_genres])
            elem_plot = plot_elems.find('p').find_all('span')
            plot = elem_plot[-1].text.strip()
        except Exception as e:
            try:
                plot_elems = soup.find('div', class_='sc-9a2a0028-10 dFokEJ')
                elem_genres = plot_elems.find('div', class_='ipc-chip-list__scroller').find_all('a')
                genres = ', '.join([a.text.strip() for a in elem_genres])
                genre_refs = ', '.join([imdb_url + a['href'] for a in elem_genres])
                elem_plot = plot_elems.find('p').find_all('span')
                plot = elem_plot[-1].text.strip()
            except Exception as e:
                error_log = f'Plot block exception: {e}, Title: {title}'
                print(error_log)
                write_error_log(ind, url, error_log)
                genres, plot, genre_refs = '', '', ''

        try:
            rate_elems = soup.find('div', class_='sc-9a2a0028-11 ketnsO')
            try:
                rating = rate_elems.find('div', 'sc-d541859f-2 kxphVf')
                imdb = float(rating.find_all('span')[0].text.strip())
            except:
                imdb = ''
            try:
                rate = rate_elems.find('ul', 'ipc-inline-list sc-b782214c-0 bllRjU baseAlt')
                metascore = int(rate.find_all('li')[2].find('span',class_='score').text.strip())
            except:
                metascore = ''
        except Exception as e:
            #print(f'Rating block exception: {e}, Title: {title}')
            imdb, metascore = '', ''
        try:
            award_elems = soup.find('section', class_='ipc-page-section',
                                     attrs = {'cel_widget_id':'StaticFeature_Awards'})
            award_refs = imdb_url + award_elems.find('ul').find('li').find('a')['href']
        except Exception:
            award_refs = ''
        try:
            ep_elems = soup.find('section', class_='ipc-page-section',
                                  attrs = {'data-testid':'Episodes'})
            epi_elems = ep_elems.find('h3')
            if epi_elems:
                epi_refs = imdb_url + ep_elems.find('div', class_='ipc-title').find('a')['href']
                episodes = int(epi_elems.find_all('span')[-1].text.strip())
            else:
                episodes, epi_refs  = '', ''
        except Exception as e:
            #print(f'Episode block exception: {e}, Title: {title}')
            episodes, epi_refs  = '', ''
        try:
            cast_elems = soup.find('section', class_='ipc-page-section',
                                    attrs = {'data-testid':'title-cast'})
            if cast_elems:
                cast_refs = imdb_url + cast_elems.find('div', class_='ipc-title__wrapper').find('a')['href']
            else:
                cast_refs = ''
        except Exception as e:
            error_log = f'Cast block exception: {e}, Title: {title}'
            print(error_log)
            write_error_log(ind, url, error_log)
            cast_refs = ''
    except Exception as e:
        error_log = f'Movie block exception: {e}'
        print(error_log)
        write_error_log(ind, url, error_log)
        title, movie_type, year, age_limit, m_time, genres, plot, episodes = ['']*8
        epi_refs, cast_refs, genre_refs, imdb, orig_title, award_refs, metascore = ['']*7


    d = {'url':url, 'title': title, 'orig_title': orig_title, 'movie_type': movie_type,
         'year': year, 'age_limit': age_limit, 'm_time': m_time, 'genres': genres,
         'episodes': episodes, 'imdb': imdb, 'metascore':metascore,
         'epi_refs': epi_refs, 'cast_refs': cast_refs, 'genre_refs': genre_refs,
         'award_refs':award_refs, 'plot': plot}
    #print(d)
    write_title_data(d)

#continue

In [ ]:
df = pd.read_csv('imdb_movies.csv', names = ['url', 'title']+[str(i) for i in range(14)])
df.tail()

,url,title,0,1,2,3,4,5,6,7,8,9,10,11,12,13
103654,https://www.imdb.com/title/tt14039724/?ref_=sr...,Ichata Vahanamulu Nilupa Radu,Ichata Vahanumulu Niluparadu,Movie,2021,NaN,2h 31m,"Drama, Mystery, Romance, Thriller",NaN,5.7,NaN,NaN,https://www.imdb.com/title/tt14039724/fullcred...,https://www.imdb.com/interest/in0000076/?ref_=...,https://www.imdb.com/title/tt14039724/awards/?...,"Arun is an architect who has a loving mother, ..."
103655,https://www.imdb.com/title/tt30472557/?ref_=sr...,Chainsaw Man - The Movie: Reze Arc,Gekijô-ban Chensô Man Reze-hen,Movie,2025,NaN,NaN,"Adult Animation, Anime, Shōnen, Supernatural F...",NaN,NaN,NaN,NaN,https://www.imdb.com/title/tt30472557/fullcred...,https://www.imdb.com/interest/in0000025/?ref_=...,NaN,"Denji encounters a new romantic interest, Reze..."
103656,https://www.imdb.com/title/tt1185616/?ref_=sr_...,Waltz with Bashir,Vals Im Bashir,Movie,2008,R,1h 30m,"Adult Animation, Computer Animation, Docudrama...",NaN,8.0,91.0,NaN,https://www.imdb.com/title/tt1185616/fullcredi...,https://www.imdb.com/interest/in0000025/?ref_=...,https://www.imdb.com/title/tt1185616/awards/?r...,An Israeli film director interviews fellow vet...
103657,https://www.imdb.com/title/tt9182964/?ref_=sr_...,The Reckoning,NaN,Movie,2020,Not Rated,1h 50m,"Adventure, Drama, Horror",NaN,4.8,31.0,NaN,https://www.imdb.com/title/tt9182964/fullcredi...,https://www.imdb.com/interest/in0000012/?ref_=...,https://www.imdb.com/title/tt9182964/awards/?r...,"Grace, a young widow haunted by the recent sui..."
103658,https://www.imdb.com/title/tt0058479/?ref_=sr_...,The Pleasure Seekers,NaN,Movie,1964,Approved,1h 47m,"Comedy, Musical, Romance",NaN,NaN,NaN,NaN,https://www.imdb.com/title/tt0058479/fullcredi...,https://www.imdb.com/interest/in0000034/?ref_=...,https://www.imdb.com/title/tt0058479/awards/?r...,Three American lovelies rooming together in Ma...


In [ ]:
df[df.title == "The Fox"]

,url,title,0,1,2,3,4,5,6,7,8,9,10,11,12,13
5004,https://www.imdb.com/title/tt14439178/?ref_=sr...,The Fox,Der Fuchs,Movie,2022,NaN,1h 58m,"Drama, History, War",NaN,7.1,NaN,NaN,https://www.imdb.com/title/tt14439178/fullcred...,https://www.imdb.com/interest/in0000076/?ref_=...,https://www.imdb.com/title/tt14439178/awards/?...,"At the dawn of WW2, a young soldier encounters..."
13648,https://www.imdb.com/title/tt12525406/?ref_=sr...,The Fox,NaN,NaN,NaN,NaN,NaN,Comedy,NaN,NaN,NaN,NaN,https://www.imdb.com/title/tt12525406/fullcred...,https://www.imdb.com/interest/in0000034/?ref_=...,NaN,"In this black comedic folktale, an affable fox..."
73803,https://www.imdb.com/title/tt14439178/?ref_=sr...,The Fox,Der Fuchs,Movie,2022,NaN,1h 58m,"Drama, History, War",NaN,7.1,NaN,NaN,https://www.imdb.com/title/tt14439178/fullcred...,https://www.imdb.com/interest/in0000076/?ref_=...,https://www.imdb.com/title/tt14439178/awards/?...,"At the dawn of WW2, a young soldier encounters..."


In [ ]:
df['13'].str.len().max()

382.0

In [ ]:
df['10'].value_counts()

,count
10,
https://www.imdb.com/title/tt32384391/fullcredits/?ref_=tt_cst_sm,11
https://www.imdb.com/title/tt3061046/fullcredits/?ref_=tt_cst_sm,11
https://www.imdb.com/title/tt1710308/fullcredits/?ref_=tt_cst_sm,10
https://www.imdb.com/title/tt2742544/fullcredits/?ref_=tt_cst_sm,10
https://www.imdb.com/title/tt11165358/fullcredits/?ref_=tt_cst_sm,10
...,...
https://www.imdb.com/title/tt0031619/fullcredits/?ref_=tt_cst_sm,1
https://www.imdb.com/title/tt21998526/fullcredits/?ref_=tt_cst_sm,1
https://www.imdb.com/title/tt6982604/fullcredits/?ref_=tt_cst_sm,1


In [ ]:
df[df['9'].str.len()==df['9'].str.len().max()]

AttributeError: Can only use .str accessor with string values!

In [ ]:
genre_urls = []
for i in range(len(df)):
    if str(df['11'][i]) != 'nan':
        urls = df['11'][i].split(', ')
        for u in urls:
            url = u.split('?')[0]
            if url not in genre_urls:
                genre_urls.append(url)

In [ ]:
genre_urls[:10], len(genre_urls), len(set(genre_urls))

(['https://www.imdb.com/interest/in0000177/',
  'https://www.imdb.com/interest/in0000159/',
  'https://www.imdb.com/interest/in0000001/',
  'https://www.imdb.com/interest/in0000076/',
  'https://www.imdb.com/interest/in0000162/',
  'https://www.imdb.com/interest/in0000186/',
  'https://www.imdb.com/interest/in0000152/',
  'https://www.imdb.com/interest/in0000052/',
  'https://www.imdb.com/interest/in0000139/',
  'https://www.imdb.com/interest/in0000026/'],
 212,
 212)

In [ ]:
set_genres =  list(set(genre_urls))
with open('genre_urls.txt', 'w') as f:
    for i in set_genres:
        f.write(i + '\n')

#genres

In [ ]:
with open('genre_urls.txt', 'r') as f:
    genre_urls = f.readlines()
genre_urls = [u.strip() for u in genre_urls]

In [ ]:
def write_genre_data(data):
    with open('imdb_genres.csv','a') as f:
        writer = csv.writer(f)
        writer.writerow((
            data['url'],
            data['genre'],
            data['descr']
            ))

In [ ]:
def write_genre_error_log(i, url, error_log):
    with open('error_genre_log.txt', 'a') as f:
        f.write(f'{i} | {url} | {error_log} \n')
    with open('error_genre_urls.txt', 'a') as f:
        f.write(url + '\n')

In [ ]:
def get_genre_data(html, ind, url):
    imdb_url = 'https://www.imdb.com'
    soup = BeautifulSoup(html, 'html.parser', from_encoding='utf-8')
    try:
        block = soup.find('div', class_='ipc-page-content-container ipc-page-content-container--center sc-986b349d-0 gBOYNf')
        block_d = block.find('div', class_='sc-81d15d15-0 cOmmee')
        genre_elem = block_d.find('div', class_='sc-81d15d15-2 kDKCev')
        try:
            genre = genre_elem.find('div', class_='ipc-title').find('h3').text.strip()
        except Exception as e:
            error_log = f'Genre name block exception: {e}, url: {url}'
            print(error_log)
            write_genre_error_log(ind, url, error_log)
            genre = ''
        try:
            descr = genre_elem.find('div', class_='ipc-overflowText').find('div', class_='ipc-html-content-inner-div').text.strip()
        except Exception as e:
            error_log = f'Description block exception: {e}, Genre: {genre}'
            print(error_log)
            write_genre_error_log(ind, url, error_log)
            descr = ''
    except Exception as e:
        error_log = f'Content block Exception: {e}, url: {url}'
        print(error_log)
        write_genre_error_log(ind, url, error_log)
        genre, descr = '', ''
    d = {'url': url, 'genre':genre, 'descr':descr}
    #print(d)
    write_genre_data(d)

In [ ]:
#drama_titles.append('https://www.imdb.com/title/tt0808357/?ref_=nv_sr_srsg_0_tt_8_nm_0_in_0_q_lust')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
for i in range(len(genre_urls)):
    url = genre_urls[i]
    response = requests.get(url,
                            headers={'Accept-Language': 'en-US,en;q=0.9',
                                          #'User-Agent': UserAgent().chrome
                                     'User-Agent':user_agent})
    #print(response.headers.get('Content-Type'))
    if i % 10 == 0:
        print(url)
    html = response.content.decode('utf-8')
    get_genre_data(html, i, url)
    time.sleep(1)

https://www.imdb.com/interest/in0000180/
https://www.imdb.com/interest/in0000105/
https://www.imdb.com/interest/in0000052/
https://www.imdb.com/interest/in0000135/
https://www.imdb.com/interest/in0000129/
https://www.imdb.com/interest/in0000061/
https://www.imdb.com/interest/in0000068/
https://www.imdb.com/interest/in0000087/
https://www.imdb.com/interest/in0000041/
https://www.imdb.com/interest/in0000028/
https://www.imdb.com/interest/in0000099/
https://www.imdb.com/interest/in0000152/
https://www.imdb.com/interest/in0000163/
Content block Exception: 'NoneType' object has no attribute 'find', url: https://www.imdb.com/search/title/
https://www.imdb.com/interest/in0000190/
https://www.imdb.com/interest/in0000128/
https://www.imdb.com/interest/in0000054/
https://www.imdb.com/interest/in0000079/
https://www.imdb.com/interest/in0000022/
https://www.imdb.com/interest/in0000091/
https://www.imdb.com/interest/in0000116/
https://www.imdb.com/interest/in0000168/
https://www.imdb.com/interest/i

#Cast

##prep

In [ ]:
df = pd.read_csv('imdb_movies.csv', names = ['url', 'title']+[str(i) for i in range(14)])
df.tail(3)

,url,title,0,1,2,3,4,5,6,7,8,9,10,11,12,13
103656,https://www.imdb.com/title/tt1185616/?ref_=sr_...,Waltz with Bashir,Vals Im Bashir,Movie,2008,R,1h 30m,"Adult Animation, Computer Animation, Docudrama...",NaN,8.0,91.0,NaN,https://www.imdb.com/title/tt1185616/fullcredi...,https://www.imdb.com/interest/in0000025/?ref_=...,https://www.imdb.com/title/tt1185616/awards/?r...,An Israeli film director interviews fellow vet...
103657,https://www.imdb.com/title/tt9182964/?ref_=sr_...,The Reckoning,NaN,Movie,2020,Not Rated,1h 50m,"Adventure, Drama, Horror",NaN,4.8,31.0,NaN,https://www.imdb.com/title/tt9182964/fullcredi...,https://www.imdb.com/interest/in0000012/?ref_=...,https://www.imdb.com/title/tt9182964/awards/?r...,"Grace, a young widow haunted by the recent sui..."
103658,https://www.imdb.com/title/tt0058479/?ref_=sr_...,The Pleasure Seekers,NaN,Movie,1964,Approved,1h 47m,"Comedy, Musical, Romance",NaN,NaN,NaN,NaN,https://www.imdb.com/title/tt0058479/fullcredi...,https://www.imdb.com/interest/in0000034/?ref_=...,https://www.imdb.com/title/tt0058479/awards/?r...,Three American lovelies rooming together in Ma...


In [ ]:
dt = df[['10', '1', '2']].copy()
dt.copy()

,10,1,2
0,https://www.imdb.com/title/tt0113481/fullcredi...,Movie,1995
1,https://www.imdb.com/title/tt1982229/fullcredi...,TV Series,2011
2,https://www.imdb.com/title/tt0041699/fullcredi...,Movie,1949
3,https://www.imdb.com/title/tt14344936/fullcred...,Movie,2020
4,https://www.imdb.com/title/tt0078490/fullcredi...,Movie,1978
...,...,...,...
103654,https://www.imdb.com/title/tt14039724/fullcred...,Movie,2021
103655,https://www.imdb.com/title/tt30472557/fullcred...,Movie,2025
103656,https://www.imdb.com/title/tt1185616/fullcredi...,Movie,2008
103657,https://www.imdb.com/title/tt9182964/fullcredi...,Movie,2020


In [ ]:
dt = dt[(dt['1'].isna()==False)&(dt['10'].isna()==False)]

In [ ]:
dt = dt.drop_duplicates(keep='first')

In [ ]:
dt.to_csv('cast_urls.csv', index=False, header=False)

In [ ]:
dt['10'].str.split('?')

In [ ]:
cast_urls = [url.split('?')[0] for url in df['10'].values if url is not np.nan]

In [ ]:
cast_urls = list(set(cast_urls))

##parser

In [ ]:
dt = pd.read_csv('cast_urls.csv', names=['url', 'mtype', 'year'])
dt.head(3)

,url,mtype,year
0,https://www.imdb.com/title/tt0113481/fullcredi...,Movie,1995
1,https://www.imdb.com/title/tt1982229/fullcredi...,TV Series,2011
2,https://www.imdb.com/title/tt0041699/fullcredi...,Movie,1949


In [ ]:
def write_cast_data(data):
    with open('imdb_casts.csv','a') as f:
        writer = csv.writer(f)
        writer.writerow((
            data['url'],
            data['title'],
            data['department'],
            data['name'],
            data['position'],
            data['comment'],
            data['episodes'],
            data['year'],
            data['credited'],
            data['rating'],
            data['name_ref']
            ))

In [ ]:
def write_cast_error_log(i, url, error_log):
    with open('error_cast_log.txt', 'a') as f:
        f.write(f'{i} | {url} | {error_log} \n')
    with open('error_cast_urls.txt', 'a') as f:
        f.write(url + '\n')

In [ ]:
def separated_position(position):
    """
    Separates position, comment and credited from a string

    Args:
        position (str): full line of position/role.

    Returns:
       data (list): list of dicts in separated lines
    """
    #Checking whether person worked for several positions
    data = []
    comment = ''
    if '/ ...' in position:
        position = position.replace('/ ...', '').strip()
    if ' / ' not in position:
        if ':' in position:
            pos = position.split(':')[0]
            comment = ', '.join(position.split(':')[1:])
            position, comment = pos.strip(), comment.strip()
        position, comment, credited = separated_position_comments_credits(position, comment)
        if 'uncredited' in comment.lower():
            comment = comment.replace('uncredited', '').replace('(', '').replace(')', '').strip()
            credited = 'Uncredited'
        elif '(as ' in comment.lower():
            comment, credited = comment.split('(as ')
            credited = credited.replace(')', '').strip()
            comment = comment.strip()
        data.append({'position':position, 'comment':comment, 'credited':credited})
    else:
        sep_position = position.split(' / ')
        for pos in sep_position:
            if ':' in pos:
                position = pos.split(':')[0]
                comment = ', '.join(pos.split(':')[1:])
                position, comment = position.strip(), comment.strip()
                position, comment, credited = separated_position_comments_credits(position, comment)
                if 'uncredited' in comment.lower():
                    comment = comment.replace('uncredited', '').replace('(', '').replace(')', '').strip()
                    credited = 'Uncredited'
                elif '(as ' in comment.lower():
                    comment, credited = comment.split('(as ')
                    credited = credited.replace(')', '').strip()
                    comment = comment.strip()
                data.append({'position':position, 'comment':comment, 'credited':credited})
            else:
                position, comment = pos.strip(), ''
                position, comment, credited = separated_position_comments_credits(position, comment)
                if 'uncredited' in comment.lower():
                    comment = comment.replace('uncredited', '').replace('(', '').replace(')', '').strip()
                    credited = 'Uncredited'
                elif '(as ' in comment.lower():
                    comment, credited = comment.split('(as ')
                    credited = credited.replace(')', '').strip()
                    comment = comment.strip()
                data.append({'position':position, 'comment':comment, 'credited':credited})
    return data

def separated_position_comments_credits(position, comment):
    credited = ''
    #Checking whether the person is uncredited
    if 'uncredited' in position.lower():
        position = position.replace('uncredited', '').replace('(', '').replace(')', '').strip()
        credited = 'Uncredited'
    #Checking whether the person is credited under other name
    elif '(as ' in position.lower():
        position, credited = position.split('(as ')
        credited = credited.replace(')', '').strip()
        position = position.strip()
    if '(' in position:
        position = position.replace(')', '').replace('(', '').strip()
    else:
        position = position.strip()
    return position, comment, credited

def separated_episodes(episodes, year):
    if 'unknown episodes' not in episodes:
        try:
            episodes, year = episodes.split(', ')
            episodes = episodes.replace('(', '').replace(')', '').strip()
            year = year.replace('(', '').replace(')', '').strip()
        except Exception as e:
            episodes = episodes.replace('(', '').replace(')', '').strip()
            year = year.replace('(', '').replace(')', '').strip()
    else:
        episodes = 'unknown episodes'
    return episodes, year

In [ ]:
def parse_series(cast_table, cast_h4, imdb_url, title, year, ind):
    full_list = []
    for i in range(len(cast_table)):
        h4, table = cast_h4[i], cast_table[i]
        department = h4.text.strip().replace('\n      ', '').replace('\n', '')
        info = table.find_all('tr')
        if 'Cast' in department and 'Casting' not in department:
            for j in range(1, len(info)):
                tds = info[j].find_all('td')
                if len(tds)>1:
                    rating = j
                    try:
                        name = tds[1].find('a').text.strip()
                        name_ref = imdb_url + tds[1].find('a')['href']
                    except Exception as e:
                        error_msg = f'Name block exception: url={url}, dep={department} | {e}'
                        write_cast_error_log(ind, url, error_msg)
                        print(ind, error_msg)
                    try:
                        href = tds[3].find_all('a')
                        episodes = href[1].text
                        position = tds[3].text.replace(episodes, '')
                        episodes, year = separated_episodes(episodes, year)
                        data = separated_position(position)
                    except Exception as e:
                        try:
                            href = tds[3].find_all('a')
                            episodes = href[0].text
                            data = [{'position':'', 'comment':'', 'credited':''}]
                            episodes, year = separated_episodes(episodes, year)
                        except Exception as e:
                            error_msg = f'Position block exception: url={url}, dep={department}, name={name} | {e}'
                            write_cast_error_log(ind, url, error_msg)
                            print(ind, error_msg)
                            data = [{'position':'', 'comment':'', 'credited':''}]
                            episodes = ''
                    for dct in data:
                        d = {'url':url, 'title':title, 'department': department,
                            'name':name, 'position':dct['position'],
                            'comment': dct['comment'], 'episodes':episodes,
                            'year':year, 'credited': dct['credited'],
                            'rating':rating, 'name_ref':name_ref}
                        full_list.append(d)
                else:
                    continue
        else:
            for j in range(len(info)):
                rating = j + 1
                tds = info[j].find_all('td')
                name = tds[0].find('a').text.strip()
                name_ref = imdb_url + tds[0].find('a')['href']
                full_position = tds[2].text.strip()
                try:
                    m = re.search(r'\(*\d* episodes*,* *\d*-*\d*\)*', full_position)
                    if m:
                        episodes = m.group(0)
                    elif '(unknown episodes)' in full_position:
                        episodes = 'unknown episodes'
                    else:
                        episodes = ''
                    position = full_position.replace(episodes, '').strip()
                    data = separated_position(position)
                    episodes, year = separated_episodes(episodes, year)
                    for dct in data:
                        d = {'url':url, 'title':title, 'department': department,
                            'name':name, 'position':dct['position'],
                            'comment': dct['comment'], 'episodes':episodes,
                            'year':year, 'credited': dct['credited'],
                            'rating':rating, 'name_ref':name_ref}
                        full_list.append(d)
                except Exception as e:
                    error_msg = f'Position block exception: url={url}, dep={department}, name={name} | {e}'
                    write_cast_error_log(ind, url, error_msg)
                    print(ind, error_msg)
                    continue
    return full_list

In [ ]:
def parse_movies(cast_table, cast_h4, imdb_url, title, year, ind):
    full_list = []
    for i in range(len(cast_table)):
        h4, table = cast_h4[i], cast_table[i]
        department = h4.text.strip().replace('\n      ', '').replace('\n', '')
        info = [tr for tr in table.find_all('tr') if tr.find('td', class_='name') is not None or\
                tr.find('td', class_='primary_photo') is not None]
        if 'Cast' in department and 'Casting' not in department:
            for j in range(len(info)):
                try:
                    row = info[j]
                    tds = row.find_all('td')
                    if len(tds)>1:
                        rating = j
                        name = tds[1].find('a').text.strip()
                        name_ref = imdb_url + tds[1].find('a')['href']
                        try:
                            position = tds[3].text.strip()
                            episodes = ''
                        except Exception as e:
                            position, episodes = '', ''
                        data = separated_position(position)
                        for dct in data:
                            d = {'url':url, 'title':title, 'department': department,
                                'name':name, 'position':dct['position'],
                                'comment': dct['comment'], 'episodes':episodes,
                                'year':year, 'credited': dct['credited'],
                                'rating':rating, 'name_ref':name_ref}
                            full_list.append(d)
                    else:
                        continue
                except Exception as e:
                    error_msg = f'Movie Actors Exception: url={url}, dep={department}, name={name} | {e}'
                    write_cast_error_log(ind, url, error_msg)
                    print(ind, error_msg)
                    continue
        else:
            for j in range(len(info)):
                try:
                    rating = j + 1
                    tds = info[j].find_all('td')
                    name = tds[0].find('a').text.strip()
                    name_ref = imdb_url + tds[0].find('a')['href']
                    try:
                        position = tds[2].text.strip()
                    except Exception as e:
                        position = ''
                    episodes = ''
                    data = separated_position(position)
                    for dct in data:
                        d = {'url':url, 'title':title, 'department': department,
                            'name':name, 'position':dct['position'],
                            'comment': dct['comment'], 'episodes':episodes,
                            'year':year, 'credited': dct['credited'],
                            'rating':rating, 'name_ref':name_ref}
                        full_list.append(d)
                except Exception as e:
                    error_msg = f'Movie Cast Exception: url={url}, dep={department}, name={name} | {e}'
                    write_cast_error_log(ind, url, error_msg)
                    print(ind, error_msg)
                    continue
    return full_list


In [ ]:
def get_cast_data(html, url, mtype, year, ind):
    imdb_url = 'https://www.imdb.com'
    soup = BeautifulSoup(html, 'html.parser', from_encoding='utf-8')
    try:
        main = soup.find('div', attrs={'id':'wrapper'}).find('div', attrs={'id':'root'}).find('div', attrs={'id':'main'})
        try:
            title = main.find('div', class_='subpage_title_block').find('h3').find('a').text.strip()
        except Exception as e:
            print(f'Title block Exception: url={url}')
            title = ''
        try:
            cast_elems = main.find('div', class_='header', attrs={'id':'fullcredits_content'})
            cast_h4 = cast_elems.find_all('h4')
            cast_table = cast_elems.find_all('table')
        except Exception as e:
            error_msg = f'Cast block Exception: url={url}, msg={e}'
            write_cast_error_log(ind, url, error_msg)
            print(ind, error_msg)
        if mtype in ['TV Series', 'TV Mini Series']:
            full_list = parse_series(cast_table, cast_h4, imdb_url, title, year, ind)
        else:
            full_list = parse_movies(cast_table, cast_h4, imdb_url, title, year, ind)
        for data in full_list:
            write_cast_data(data)
    except Exception as e:
        error_msg = f'Content block Exception: url={url}, e={e}'
        write_cast_error_log(ind, url, error_msg)
        print(ind, error_msg)

#parsed

In [ ]:
def write_cast_data(data):
    with open('imdb_casts_unparsed_50000.csv','a') as f:
        writer = csv.writer(f)
        writer.writerow((
            data['url'],
            data['title'],
            data['department'],
            data['name'],
            data['position'],
            data['comment'],
            data['episodes'],
            data['year'],
            data['credited'],
            data['rating'],
            data['name_ref']
            ))

In [ ]:
def separated_position(position):
    """
    Separates position, comment and credited from a string

    Args:
        position (str): full line of position/role.

    Returns:
       data (list): list of dicts in separated lines
    """
    #Checking whether person worked for several positions
    data = []
    comment = ''
    if '/ ...' in position:
        position = position.replace('/ ...', '').strip()
    if ' / ' not in position:
        if ':' in position:
            pos = position.split(':')[0]
            comment = ', '.join(position.split(':')[1:])
            position, comment = pos.strip(), comment.strip()
        position, comment, credited = separated_position_comments_credits(position, comment)
        if 'uncredited' in comment.lower():
            comment = comment.replace('uncredited', '').replace('(', '').replace(')', '').strip()
            credited = 'Uncredited'
        elif '(as ' in comment.lower():
            if comment.count('(as') == 1:
                comment, credited = comment.split('(as ')
                credited = credited.replace(')', '').strip()
                comment = comment.strip()
            elif comment.count('(as') == 2:
                comment, credited = comment.split('(as ')[0], ', '.join([com.strip() for com in comment.split('(as')[1:]])
                credited = credited.replace(')', '').strip()
                comment = comment.strip()
        data.append({'position':position, 'comment':comment, 'credited':credited})
    else:
        sep_position = position.split(' / ')
        for pos in sep_position:
            if ':' in pos:
                position = pos.split(':')[0]
                comment = ', '.join(pos.split(':')[1:])
                position, comment = position.strip(), comment.strip()
                position, comment, credited = separated_position_comments_credits(position, comment)
                if 'uncredited' in comment.lower():
                    comment = comment.replace('uncredited', '').replace('(', '').replace(')', '').strip()
                    credited = 'Uncredited'
                elif '(as ' in comment.lower():
                    if comment.count('(as') == 1:
                        comment, credited = comment.split('(as ')
                        credited = credited.replace(')', '').strip()
                        comment = comment.strip()
                    elif comment.count('(as') == 2:
                        comment, credited = comment.split('(as ')[0], ', '.join([com.strip() for com in comment.split('(as')[1:]])
                        credited = credited.replace(')', '').strip()
                        comment = comment.strip()
                data.append({'position':position, 'comment':comment, 'credited':credited})
            else:
                position, comment = pos.strip(), ''
                position, comment, credited = separated_position_comments_credits(position, comment)
                if 'uncredited' in comment.lower():
                    comment = comment.replace('uncredited', '').replace('(', '').replace(')', '').strip()
                    credited = 'Uncredited'
                elif '(as ' in comment.lower():
                    if comment.count('(as') == 1:
                        comment, credited = comment.split('(as ')
                        credited = credited.replace(')', '').strip()
                        comment = comment.strip()
                    elif comment.count('(as') == 2:
                        comment, credited = comment.split('(as ')[0], ', '.join([com.strip() for com in comment.split('(as')[1:]])
                        credited = credited.replace(')', '').strip()
                        comment = comment.strip()
                data.append({'position':position, 'comment':comment, 'credited':credited})
    return data

def separated_position_comments_credits(position, comment):
    credited = ''
    #Checking whether the person is uncredited
    if 'uncredited' in position.lower():
        position = position.replace('uncredited', '').replace('(', '').replace(')', '').strip()
        credited = 'Uncredited'
    #Checking whether the person is credited under other name
    elif '(as ' in position:
        if position.count('(as') == 1:
            position, credited = position.split('(as ')
            credited = credited.replace(')', '').strip()
            position = position.strip()
        elif position.count('(as') == 2:
            position, credited = position.split('(as ')[0], ', '.join([com.strip() for com in position.split('(as')[1:]])
            credited = credited.replace(')', '').strip()
            position = position.strip()
    elif '(As ' in position:
        position, credited = position.split('(As ')
        credited = credited.replace(')', '').strip()
        position = position.strip()
    elif 'singing voice' in position and 'additional' not in position:
        position = position.replace('singing voice', '').strip()
        comment = 'singing voice'
    elif 'voice' in position and 'additional' not in position:
        position = position.replace('voice', '').strip()
        comment = 'voice'


    if '(' in position:
        position = position.replace(')', '').replace('(', '').strip()
    else:
        position = position.strip()
    return position, comment, credited

def separated_episodes(episodes, year):
    if 'unknown episodes' not in episodes:
        try:
            episodes, year = episodes.split(', ')
            episodes = episodes.replace('(', '').replace(')', '').strip()
            year = year.replace('(', '').replace(')', '').strip()
        except Exception as e:
            episodes = episodes.replace('(', '').replace(')', '').strip()
            year = year.replace('(', '').replace(')', '').strip()
    else:
        episodes = 'unknown episodes'
    return episodes, year

In [ ]:
def parse_series(cast_table, cast_h4, imdb_url, title, year, ind):
    full_list = []
    for i in range(len(cast_h4)):
        h4, table = cast_h4[i], cast_table[i]
        department = h4.text.strip().replace('\n      ', '').replace('\n', '')
        info = table.find_all('tr')
        #print(department)
        if 'Cast' in department and 'Casting' not in department:
            for j in range(1, len(info)):
                tds = info[j].find_all('td')
                if len(tds)>1:
                    rating = j
                    try:
                        name = tds[1].find('a').text.strip()
                        name_ref = imdb_url + tds[1].find('a')['href']
                    except Exception as e:
                        error_msg = f'Name block exception: url={url}, dep={department} | {e}'
                        write_cast_error_log(ind, url, error_msg)
                        print(ind, error_msg)
                    try:
                        href = tds[3].find_all('a')
                        #print(href)
                        episodes = ''

                        if href[0].text.strip() != href[-1].text.strip():
                            full_position = href[0].text.strip() +  ' ' + href[-1].text.strip()
                        else:
                            full_position = href[-1].text
                        #print(full_position)
                        # episodes, year = separated_episodes(episodes, year)
                        # data = separated_position(position)
                        try:
                            m = re.search(r'\(*\d* episodes*,* *\d*-*\d*\)*', full_position)
                            y = re.search(r'\(*\d{4}\)*', full_position)
                            if '(unknown episodes)' in full_position or 'unknown episodes' in full_position:
                                episodes = 'unknown episodes'
                            elif m:
                                episodes = m.group(0)
                            elif y:
                                episodes = ''
                                year = y.group(0)
                                full_position = full_position.replace(year, '').strip()
                            else:
                                episodes = ''
                            position = full_position.replace(episodes, '').strip()
                            data = separated_position(position)
                            episodes, year = separated_episodes(episodes, year)
                            for dct in data:
                                d = {'url':url, 'title':title, 'department': department,
                                    'name':name, 'position':dct['position'],
                                    'comment': dct['comment'], 'episodes':episodes,
                                    'year':year, 'credited': dct['credited'],
                                    'rating':rating, 'name_ref':name_ref}
                                full_list.append(d)
                                #print(d)
                        except Exception as e:
                            error_msg = f'Position block exception: url={url}, dep={department}, name={name} | {e}'
                            write_cast_error_log(ind, url, error_msg)
                            print(ind, error_msg)
                            continue

                    except Exception as e:
                        try:
                            href = tds[3].find_all('a')
                            episodes = href[0].text
                            data = [{'position':'', 'comment':'', 'credited':''}]
                            episodes, year = separated_episodes(episodes, year)
                        except Exception as e:
                            error_msg = f'Position block exception: url={url}, dep={department}, name={name} | {e}'
                            write_cast_error_log(ind, url, error_msg)
                            print(ind, error_msg)
                            data = [{'position':'', 'comment':'', 'credited':''}]
                            episodes = ''
                        for dct in data:
                            d = {'url':url, 'title':title, 'department': department,
                                'name':name, 'position':dct['position'],
                                'comment': dct['comment'], 'episodes':episodes,
                                'year':year, 'credited': dct['credited'],
                                'rating':rating, 'name_ref':name_ref}
                            full_list.append(d)
                            #print(d)
                else:
                    continue
        else:
            for j in range(len(info)):
                rating = j + 1
                tds = info[j].find_all('td')
                name = tds[0].find('a').text.strip()
                name_ref = imdb_url + tds[0].find('a')['href']
                try:
                    full_position = tds[2].text.strip()
                    try:
                        m = re.search(r'\(*\d* episodes*,* *\d*-*\d*\)*', full_position)
                        y = re.search(r'\(*\d{4}\)*', full_position)
                        if '(unknown episodes)' in full_position or 'unknown episodes' in full_position:
                            episodes = 'unknown episodes'
                        elif m:
                            episodes = m.group(0)
                        elif y:
                            episodes = ''
                            year = y.group(0)
                            full_position = full_position.replace(year, '').strip()
                        else:
                            episodes = ''
                        position = full_position.replace(episodes, '').strip()
                        data = separated_position(position)
                        episodes, year = separated_episodes(episodes, year)
                        for dct in data:
                            d = {'url':url, 'title':title, 'department': department,
                                'name':name, 'position':dct['position'],
                                'comment': dct['comment'], 'episodes':episodes,
                                'year':year, 'credited': dct['credited'],
                                'rating':rating, 'name_ref':name_ref}
                            full_list.append(d)
                            #print(d)
                    except Exception as e:
                        error_msg = f'Position block exception: url={url}, dep={department}, name={name} | {e}'
                        write_cast_error_log(ind, url, error_msg)
                        print(ind, error_msg)
                        continue
                except Exception:
                    d = {'url':url, 'title':title, 'department': department,
                         'name':name, 'position':'', 'comment': '', 'episodes':'',
                         'year':year, 'credited': '', 'rating':rating, 'name_ref':name_ref}
                    full_list.append(d)
    #print(full_list)
    return full_list

In [ ]:
def parse_movies(cast_table, cast_h4, imdb_url, title, year, ind):
    full_list = []
    for i in range(len(cast_table)):
        h4, table = cast_h4[i], cast_table[i]
        department = h4.text.strip().replace('\n      ', '').replace('\n', '')
        info = [tr for tr in table.find_all('tr') if tr.find('td', class_='name') is not None or\
                tr.find('td', class_='primary_photo') is not None]
        if 'Cast' in department and 'Casting' not in department:
            for j in range(len(info)):
                try:
                    row = info[j]
                    tds = row.find_all('td')
                    if len(tds)>1:
                        rating = j
                        name = tds[1].find('a').text.strip()
                        name_ref = imdb_url + tds[1].find('a')['href']
                        try:
                            position = tds[3].text.strip()
                            episodes = ''
                        except Exception as e:
                            position, episodes = '', ''
                        data = separated_position(position)
                        for dct in data:
                            d = {'url':url, 'title':title, 'department': department,
                                'name':name, 'position':dct['position'],
                                'comment': dct['comment'], 'episodes':episodes,
                                'year':year, 'credited': dct['credited'],
                                'rating':rating, 'name_ref':name_ref}
                            full_list.append(d)
                    else:
                        continue
                except Exception as e:
                    error_msg = f'Movie Actors Exception: url={url}, dep={department}, name={name} | {e}'
                    write_cast_error_log(ind, url, error_msg)
                    print(ind, error_msg)
                    continue
        else:
            for j in range(len(info)):
                try:
                    rating = j + 1
                    tds = info[j].find_all('td')
                    name = tds[0].find('a').text.strip()
                    name_ref = imdb_url + tds[0].find('a')['href']
                    try:
                        position = tds[2].text.strip()
                    except Exception as e:
                        position = ''
                    episodes = ''
                    data = separated_position(position)
                    for dct in data:
                        d = {'url':url, 'title':title, 'department': department,
                            'name':name, 'position':dct['position'],
                            'comment': dct['comment'], 'episodes':episodes,
                            'year':year, 'credited': dct['credited'],
                            'rating':rating, 'name_ref':name_ref}
                        full_list.append(d)
                except Exception as e:
                    error_msg = f'Movie Cast Exception: url={url}, dep={department}, name={name} | {e}'
                    write_cast_error_log(ind, url, error_msg)
                    print(ind, error_msg)
                    continue
    return full_list


In [ ]:
def write_cast_error_log(i, url, error_log):
    with open('error_cast_log_unp.txt', 'a') as f:
        f.write(f'{i} | {url} | {error_log} \n')
    with open('error_cast_urls_unp.txt', 'a') as f:
        f.write(url + '\n')

In [ ]:
def get_cast_data(html, url, mtype, year, ind):
    imdb_url = 'https://www.imdb.com'
    soup = BeautifulSoup(html, 'html.parser', from_encoding='utf-8')
    try:
        main = soup.find('div', attrs={'id':'main'})
        #print(main)
        try:
            title = main.find('div', class_='subpage_title_block').find('div', class_="subpage_title_block__right-column").find('h3').find('a').text.strip()
            #print(title)
        except Exception as e:
            print(f'Title block Exception: url={url}')
            title = ''
        try:
            cast_elems = main.find('div', class_='header', attrs={'id':'fullcredits_content'})
            #print(1)
            cast_h4 = cast_elems.find_all('h4')
            #print(cast_elems)
            cast_table = cast_elems.find_all('table')
        except Exception as e:
            error_msg = f'Cast block Exception: url={url}, msg={e}'
            write_cast_error_log(ind, url, error_msg)
            print(ind, error_msg)
        if mtype in ['TV Series', 'TV Mini Series']:
            full_list = parse_series(cast_table, cast_h4, imdb_url, title, year, ind)
            #print(full_list)
        else:
            full_list = parse_movies(cast_table, cast_h4, imdb_url, title, year, ind)
        for data in full_list:
           print(data)
           write_cast_data(data)
           pass
    except Exception as e:
        error_msg = f'Content block Exception: url={url}, title={title} e={e}'
        write_cast_error_log(ind, url, error_msg)
        print(ind, error_msg)

In [ ]:
#drama_titles.append('https://www.imdb.com/title/tt0808357/?ref_=nv_sr_srsg_0_tt_8_nm_0_in_0_q_lust')
#user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
ua = UserAgent()
#user_agent = ua.random
user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36'
for i in range(0, 1):
    url = 'https://www.imdb.com/title/tt3259100/fullcredits/?ref_=tt_cst_sm'
    mtype = 'TV Series'
    year = '2011– '
    response = requests.get(url,
                            headers={'Accept-Language': 'en-US,en;q=0.9',
                                     'User-Agent':user_agent})
    for key, value in response.request.headers.items():
        print(key+": "+value)
    print(i, url)
    html = response.content.decode('utf-8')
    get_cast_data(html, url, mtype, year, i)
    time.sleep(1)

NameError: name 'UserAgent' is not defined

In [ ]:
ua = UserAgent()
ua.getBrowser('google')

{'useragent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36 Edg/122.0.0.0',
 'percent': 100.0,
 'type': 'desktop',
 'device_brand': None,
 'browser': 'Edge',
 'browser_version': '122.0.0.0',
 'browser_version_major_minor': 122.0,
 'os': 'win32',
 'os_version': '10',
 'platform': 'Win32'}

In [ ]:
os.getcwd()

'/content/drive/MyDrive/Bot/Parsing'

In [ ]:
csv_files = ['imdb_casts_unparsed.csv', 'imdb_casts_unparsed_10000.csv', 'imdb_casts_unparsed_20000.csv',
             'imdb_casts_unparsed_30000.csv', 'imdb_casts_unparsed_40000.csv', 'imdb_casts_unparsed_50000.csv']
dataframes = []

# Читаем каждый CSV файл и добавляем его в список
for file in csv_files:
    df = pd.read_csv(file, names=[str(i) for i in range(11)])
    dataframes.append(df)

# Объединяем все DataFrame'ы в один
df = pd.concat(dataframes, ignore_index=True)

In [ ]:
result.to_csv('cleaned_unparsed.csv', index=False)

In [ ]:
df_present.head(50)

,0,1,2,3,4,5,6,7,8,9,10
42825,https://www.imdb.com/title/tt32525811/fullcred...,"Present, Is Present",Series Directed by,Huang Yuanda,NaN,NaN,NaN,2024-,NaN,1,https://www.imdb.com/name/nm9751809/?ref_=ttfc...
42826,https://www.imdb.com/title/tt32525811/fullcred...,"Present, Is Present",Series Cast,Tu Bing,NaN,NaN,NaN,2024-,NaN,1,https://www.imdb.com/name/nm16223494/?ref_=ttf...
42827,https://www.imdb.com/title/tt32525811/fullcred...,"Present, Is Present",Series Cast,Fan Zhi Xin,NaN,NaN,NaN,2024-,NaN,2,https://www.imdb.com/name/nm16223493/?ref_=ttf...
42828,https://www.imdb.com/title/tt32525811/fullcred...,"Present, Is Present",Series Cast,Song Yi Xiong,NaN,NaN,NaN,2024-,NaN,3,https://www.imdb.com/name/nm16223495/?ref_=ttf...


In [ ]:
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
for i in range(0, len(df)):
    url = df['url'][i]
    mtype = df['mtype'][i]
    year = df['year'][i]
    try:
        response = requests.get(url,
                                headers={'Accept-Language': 'en-US,en;q=0.9',
                                         'User-Agent':user_agent})
        print(i, url)
        html = response.content.decode('utf-8')
        get_cast_data(html, url, mtype, year, i)
    except Exception as e:
        error_msg = f'{i} | {url} | e={e}'
        print(error_msg)
        write_cast_error_log(i, url, error_msg)
    time.sleep(1)

44000 https://www.imdb.com/title/tt3846642/fullcredits/?ref_=tt_cst_sm
44001 https://www.imdb.com/title/tt7243750/fullcredits/?ref_=tt_cst_sm
44002 https://www.imdb.com/title/tt0098328/fullcredits/?ref_=tt_cst_sm
44003 https://www.imdb.com/title/tt11968256/fullcredits/?ref_=tt_cst_sm
44004 https://www.imdb.com/title/tt13183494/fullcredits/?ref_=tt_cst_sm
44005 https://www.imdb.com/title/tt0052391/fullcredits/?ref_=tt_cst_sm
44006 https://www.imdb.com/title/tt11727058/fullcredits/?ref_=tt_cst_sm
44007 https://www.imdb.com/title/tt1942887/fullcredits/?ref_=tt_cst_sm
44008 https://www.imdb.com/title/tt0236285/fullcredits/?ref_=tt_cst_sm
44009 https://www.imdb.com/title/tt21629052/fullcredits/?ref_=tt_cst_sm
44010 https://www.imdb.com/title/tt0804484/fullcredits/?ref_=tt_cst_sm
44011 https://www.imdb.com/title/tt1097239/fullcredits/?ref_=tt_cst_sm
44012 https://www.imdb.com/title/tt12787842/fullcredits/?ref_=tt_cst_sm
44013 https://www.imdb.com/title/tt0396184/fullcredits/?ref_=tt_cst_sm
4